In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import photon_number
from functions import make_title
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260414-32348-qcodes.log
Experiment loaded. Last ID no: 472


In [4]:
ID = 459
idx = 38
# datasaver.dataset.add_metadata("att_combined_id", ID)
data = load_by_id(ID).get_parameter_data()
# Extract attenuator voltages 
v_attenuator = data['v_attenuator']['v_attenuator'][idx]
avg_attenuation = data['avg_attenuation']['avg_attenuation'][idx]
avg_power90 = data['avg_power90']['avg_power90'][idx]
print(v_attenuator)

5.29999999999999


In [9]:
import scipy
import scipy.constants as spc 

# Screw attenuator calibration
ID =  165 # need to update this to read in from new config file
# datasaver.dataset.add_metadata("att_screw_calibration_id", ID)
data = load_by_id(ID).get_parameter_data()
avg_attenuation_screw = np.average(data['attenuation']['attenuation'])

# Index of attenuator voltage corresponding to ~120dB 
idx = 38 

ID = 459
data = load_by_id(ID).get_parameter_data()
v_attenuator = data['v_attenuator']['v_attenuator'][idx]
avg_attenuation = data['avg_attenuation']['avg_attenuation'][idx]
avg_power90 = data['avg_power90']['avg_power90'][idx]

# Calculate total attenuation 
total_attenuation=avg_attenuation + avg_attenuation_screw

# Check v_attenuator and total_attenuation 
print(total_attenuation)
print(v_attenuator)

Nphotons = photon_number(params.bs10, params.bs90, power90=avg_power90, total_attenuation=total_attenuation, wavelength=1550e-9)

print(Nphotons)

120.32296811465001
5.29999999999999
3413.6223253683374


In [10]:
meas = Measurement()
meas.register_custom_parameter("v_attenuator", label="V")
meas.register_custom_parameter("total_attenuation", label="dB")
meas.register_custom_parameter("Nphotons", label="")
meas.register_custom_parameter("wavelength_range", label="nm")

with meas.run() as datasaver: 
    print(datasaver.run_id)

    # Screw attenuator calibration
    ID =  165 # need to update this to read in from new config file
    # datasaver.dataset.add_metadata("att_screw_calibration_id", ID)
    data = load_by_id(ID).get_parameter_data()
    avg_attenuation_screw = np.average(data['attenuation']['attenuation'])
    
    # Index of attenuator voltage corresponding to ~120dB 
    idx = 38 
    
    ID = 459
    data = load_by_id(ID).get_parameter_data()
    v_attenuator = data['v_attenuator']['v_attenuator'][idx]
    avg_attenuation = data['avg_attenuation']['avg_attenuation'][idx]
    avg_power90 = data['avg_power90']['avg_power90'][idx]
    
    # Calculate total attenuation 
    total_attenuation=avg_attenuation + avg_attenuation_screw
    
    # Check v_attenuator and total_attenuation 
    print(total_attenuation)
    print(v_attenuator)
    
    Nphotons = photon_number(params.bs10, params.bs90, power90=avg_power90, total_attenuation=total_attenuation, wavelength=1550e-9)

    datasaver.add_result(("v_attenuator", v_attenuator),
                     ("total_attenuation", total_attenuation),
                    ("Nphotons", Nphotons))

Starting experimental run with id: 476. 
476
120.32296811465001
5.29999999999999
